In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

torch version: 2.13.0+cu130
cuda available: True


In [2]:
z = torch.tensor([2.0, 1.0, 0.1])

exp_z = torch.exp(z)
manual = exp_z / exp_z.sum()

print("取指数后:", exp_z)
print("总和    :", exp_z.sum())
print("手算结果:", manual)
print("官方结果:", torch.softmax(z, dim=0))

取指数后: tensor([7.3891, 2.7183, 1.1052])
总和    : tensor(11.2125)
手算结果: tensor([0.6590, 0.2424, 0.0986])
官方结果: tensor([0.6590, 0.2424, 0.0986])


In [3]:
a = torch.tensor([2.0, 1.0, 0.1])
b = a * 10          # 把差距放大 10 倍

print("温和输入:", torch.softmax(a, dim=0))
print("极端输入:", torch.softmax(b, dim=0))
print("平移100 :", torch.softmax(a + 100, dim=0))

温和输入: tensor([0.6590, 0.2424, 0.0986])
极端输入: tensor([9.9995e-01, 4.5398e-05, 5.6025e-09])
平移100 : tensor([0.6590, 0.2424, 0.0986])


In [4]:
def sensitivity(z):
    z = z.clone().requires_grad_(True)
    s = torch.softmax(z, dim=0)
    s[0].backward()
    return z.grad

print("温和输入:", sensitivity(torch.tensor([2.0, 1.0, 0.1])))
print("极端输入:", sensitivity(torch.tensor([20.0, 10.0, 1.0])))

温和输入: tensor([ 0.2247, -0.1598, -0.0650])
极端输入: tensor([ 4.5417e-05, -4.5396e-05, -5.6023e-09])


In [5]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (..., seq_len_q, d_k)
    K: (..., seq_len_k, d_k)
    V: (..., seq_len_k, d_v)
    mask: (..., seq_len_q, seq_len_k), 1 表示可见, 0 表示屏蔽
    返回: output (..., seq_len_q, d_v), attn_weights (..., seq_len_q, seq_len_k)
    """
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)
    output = attn_weights @ V
    return output, attn_weights

In [6]:
torch.manual_seed(0)

batch, seq_len, d_k, d_v = 2, 5, 8, 8
Q = torch.randn(batch, seq_len, d_k)
K = torch.randn(batch, seq_len, d_k)
V = torch.randn(batch, seq_len, d_v)

out, attn = scaled_dot_product_attention(Q, K, V)
print("output shape:", out.shape)      # 应为 (2, 5, 8)
print("attn shape  :", attn.shape)     # 应为 (2, 5, 5)
print("每行权重之和:", attn.sum(dim=-1))  # 应全为 1

output shape: torch.Size([2, 5, 8])
attn shape  : torch.Size([2, 5, 5])
每行权重之和: tensor([[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]])


In [7]:
causal = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)
print("mask 本体:")
print(causal[0])

out_m, attn_m = scaled_dot_product_attention(Q, K, V, mask=causal)
print("\ncausal 权重矩阵(第 0 个样本):")
print(attn_m[0])
print("\n每行仍然和为 1:", attn_m.sum(dim=-1))

mask 本体:
tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

causal 权重矩阵(第 0 个样本):
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.8355, 0.1645, 0.0000, 0.0000, 0.0000],
        [0.2639, 0.1116, 0.6245, 0.0000, 0.0000],
        [0.4357, 0.0593, 0.0602, 0.4448, 0.0000],
        [0.3215, 0.2479, 0.0755, 0.1795, 0.1757]])

每行仍然和为 1: tensor([[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]])


In [8]:
# 实验 1: 去掉缩放，把维度放大到 256
def attention_no_scale(Q, K, V):
    scores = Q @ K.transpose(-2, -1)          # 注意：没有除以 sqrt(d_k)
    return F.softmax(scores, dim=-1)

torch.manual_seed(0)
Q_big = torch.randn(1, 5, 256)
K_big = torch.randn(1, 5, 256)
V_big = torch.randn(1, 5, 256)

_, attn_scaled = scaled_dot_product_attention(Q_big, K_big, V_big)
attn_unscaled  = attention_no_scale(Q_big, K_big, V_big)

print("有缩放:")
print(attn_scaled[0])
print("\n无缩放:")
print(attn_unscaled[0])

有缩放:
tensor([[0.0076, 0.1227, 0.1095, 0.1177, 0.6425],
        [0.1154, 0.2823, 0.1427, 0.3090, 0.1505],
        [0.0816, 0.5079, 0.0925, 0.2394, 0.0786],
        [0.2518, 0.1466, 0.1767, 0.2955, 0.1294],
        [0.6771, 0.1972, 0.0434, 0.0724, 0.0099]])

无缩放:
tensor([[1.4341e-31, 3.1296e-12, 5.0557e-13, 1.6023e-12, 1.0000e+00],
        [1.1569e-07, 1.9078e-01, 3.4758e-06, 8.0920e-01, 8.1060e-06],
        [1.9492e-13, 9.9999e-01, 1.4690e-12, 5.9356e-06, 1.0742e-13],
        [7.1711e-02, 1.2492e-05, 2.4674e-04, 9.2803e-01, 1.6871e-06],
        [1.0000e+00, 2.6753e-09, 8.0363e-20, 2.9192e-16, 4.2233e-30]])


In [9]:
# 实验 2: 故意把 dim 写错
def attention_wrong_dim(Q, K, V):
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    return F.softmax(scores, dim=-2)          # 错误: 应该是 -1

_, attn_wrong = attention_wrong_dim(Q, K, V)
print("错误 dim 下，每行之和:", attn_wrong.sum(dim=-1))
print("形状完全正常:", attn_wrong.shape)

错误 dim 下，每行之和: tensor([0.7222, 0.8174, 1.0705, 1.0249, 1.3650])
形状完全正常: torch.Size([5, 5])


In [10]:
# 实验 3: 最简 self-attention
torch.manual_seed(0)
X = torch.randn(1, 4, 8)
out_self, attn_self = scaled_dot_product_attention(X, X, X)

print("权重矩阵:")
print(attn_self[0])
print("\n对角线元素:", torch.diagonal(attn_self[0]))

权重矩阵:
tensor([[0.7738, 0.0745, 0.0225, 0.1292],
        [0.1566, 0.4173, 0.0637, 0.3624],
        [0.0086, 0.0116, 0.9756, 0.0041],
        [0.1717, 0.2291, 0.0142, 0.5850]])

对角线元素: tensor([0.7738, 0.4173, 0.9756, 0.5850])


In [11]:
print("权重非负:", (attn >= 0).all().item())

# Q 全相同时，每行权重应完全一样
Q_same = torch.ones(1, 3, 8)
K_r = torch.randn(1, 3, 8)
V_r = torch.randn(1, 3, 8)
_, attn_same = scaled_dot_product_attention(Q_same, K_r, V_r)
print("Q 相同时的权重矩阵:")
print(attn_same[0])

权重非负: True
Q 相同时的权重矩阵:
tensor([[0.4203, 0.4016, 0.1781],
        [0.4203, 0.4016, 0.1781],
        [0.4203, 0.4016, 0.1781]])
